# <div align="center">🌟 Madar (مدار) - Google Colab Quickstart</div>
### <div align="center">AI-Native, High-Performance Dialect & Runtime for the Ring Ecosystem</div>

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com) 
[![GitHub](https://img.shields.io/badge/GitHub-Madar-blue.svg)](https://github.com/azzeddine2017/Madar) 
[![AVX2](https://img.shields.io/badge/CPU-AVX2%20%7C%20FMA-green.svg)]() 
[![OpenCL](https://img.shields.io/badge/GPU-OpenCL%20Zero--Copy-purple.svg)]()

---

Welcome to the official cloud quickstart notebook for **Madar (مدار)**.

This notebook is specifically configured to leverage **Google Colab's high-performance hardware**:
1. **CPU Acceleration:** 256-bit vectorized tensor kernels with `AVX2` & `FMA` SIMD instructions.
2. **GPU Acceleration:** NVIDIA Tesla GPU acceleration for FFN projection layers via zero-copy **OpenCL** (`LoadLibrary`/`dlopen` with zero external dependencies).
3. **High-Intelligence 7B Models:** Fast mounting of frontier reasoning models such as **DeepSeek-R1-7B** (Chain-of-Thought) and **Qwen2.5-7B** using zero-copy OS memory mapping (`mmap`).
4. **AlQalam C++20 Integration:** Hybrid RAG combining dense vector search (70%) with sub-millisecond lexical inverted indexing `QalamIndex` (30%).
5. **Interactive `%%madar` Cell Magic:** Write and execute Madar / Ring code directly inside notebook cells with native stdout/stderr streaming!

## ⚙️ Step 1: Verify Hardware Environment
Check CPU features (ensuring `avx2` and `fma` are present) and verify the attached NVIDIA GPU.

In [ ]:
# Check CPU architecture and SIMD flags
!lscpu | grep -E "Model name|avx2|fma"

# Check NVIDIA GPU status
!nvidia-smi

## 🔨 Step 2: Install Dependencies & Build Madar from Source
Clone the repository (if not already local) and compile the complete Madar Core runtime, **AlQalam C++20 extension**, and CLI utilities using GCC with maximum optimization flags (`-O3 -mavx2 -mfma -mf16c -pthread`).

In [ ]:
import os, sys

# 1. Clone repository if not present, or pull latest updates if already present
if not os.path.exists("/content/Madar/language"):
    !git clone https://github.com/azzeddine2017/Madar.git /content/Madar
    %cd /content/Madar
else:
    %cd /content/Madar
    !git pull

# 2. Install essential packages for OpenCL, GCC, and G++
!apt-get update -qq && apt-get install -y -qq build-essential g++ ocl-icd-opencl-dev curl libcurl4-openssl-dev

# 3. Run the complete unified Linux build pipeline
!bash tools/build_linux.sh

# 4. Verify version and runtime diagnostic specs
!bin/ring -v
!bin/ring --sysinfo

## 🪄 Step 3: Register `%%madar` and `%%ring` Cell Magic
The following cell registers a custom IPython magic command, enabling you to write native Madar/Ring code directly in any cell starting with `%%madar`.

In [ ]:
from IPython.core.magic import register_cell_magic
import subprocess, tempfile, os

MADAR_BIN = os.path.abspath("bin/ring")

@register_cell_magic
def madar(line, cell):
    """IPython Cell Magic to execute Madar code and display output directly"""
    with tempfile.NamedTemporaryFile("w", suffix=".ring", delete=False, encoding="utf-8") as f:
        f.write(cell)
        tmp_file = f.name
    
    try:
        env = os.environ.copy()
        env["LD_LIBRARY_PATH"] = os.path.abspath("lib") + ":" + env.get("LD_LIBRARY_PATH", "")
        res = subprocess.run([MADAR_BIN, tmp_file], capture_output=True, text=True, env=env)
        if res.stdout:
            print(res.stdout, end="")
        if res.stderr:
            print(res.stderr, end="", file=sys.stderr)
    finally:
        if os.path.exists(tmp_file):
            os.remove(tmp_file)

# Register %%ring as an alias
ip = get_ipython()
ip.register_magic_function(madar, 'cell', 'ring')
print("✅ %%madar and %%ring cell magics registered successfully!")

### 🧪 Quick Test: Modern Syntax Enhancements & Auto-JIT via `%%madar`

In [ ]:
%%madar
# 1. Native Scanner-level f-strings
cProject = "Madar (مدار)"
nVersion = 0.3
see f"Welcome to {cProject} v{nVersion} running on Google Colab!" + nl

# 2. Pythonic List Comprehensions with filtering
aSquares = [x*x for x in 1:10 if x % 2 == 0]
see f"Squares of even numbers: {aSquares}" + nl

# 3. Advanced Slicing (1-based, inclusive, negative stride support)
cText = "GoogleColabWithMadar"
see f"Reversed string: {cText[::-1]}" + nl

# 4. Hardware Auto-JIT Loops via SLJIT (Native CPU Machine Code)
t0 = clock()
nSum = 0
dour i = 1 to 1000000
    nSum += i
next
t1 = clock()
see f"Computed 1,000,000 iterations via JIT in: {(t1 - t0) / 1000.0} sec! Sum: {nSum}" + nl

## 🧠 Step 4: Download High-Intelligence AI Models (7B Reasoning)
With Google Colab providing **~12.7 GB RAM** and **15 GB GPU VRAM**, Madar can run 7B-parameter models:
- **DeepSeek-R1-Distill-Qwen-7B:** Advanced Chain-of-Thought (CoT) reasoning, mathematical problem solving, and code generation (~4.68 GB).
- **Qwen2.5-7B-Instruct:** SOTA general instruction-following, rich Arabic/multilingual understanding, and JSON tool calling (~4.35 GB).
- **BGE-M3:** SOTA multilingual dense embedding model for RAG (~1.19 GB).

Run the following cell to download the DeepSeek-R1-7B reasoning model (~80-100 MB/sec download speed in Colab):

In [ ]:
# Download DeepSeek-R1-7B reasoning model
!bash tools/download_colab_models.sh deepseek7b

# (Optional) To download BGE-M3 for dedicated RAG embeddings, uncomment below:
# !bash tools/download_colab_models.sh bge

## ⚡ Step 5: AI Inference with GPU Acceleration (OpenCL Zero-Copy)
Mount the 7B model using zero-copy OS memory mapping (`mmap`) and enable GPU acceleration (`model.setGPU(1)`). The GPU computes the dense FFN projections and LM Head while AVX2 multi-threading handles the attention operations.

In [ ]:
%%madar
load "stdlibCore.ring"
load "libraries/madar_ai/madar_ai.ring"

# Locate candidate model
cModelPath = "models/deepseek-r1-distill-qwen-7b-q4_0.gguf"
if not fexists(cModelPath)
    cModelPath = "models/qwen2.5-7b-instruct-q4_0.gguf"
ok

see f"[MADAR] Mounting model via zero-copy mmap: {cModelPath}" + nl
t0 = clock()
oModel = ai_open(cModelPath)
t1 = clock()
see f"[MADAR] Model mounted in: {(t1 - t0) / 1000.0} seconds!" + nl

# Configure worker threads and enable GPU acceleration
oModel.setThreads(4)
oModel.setGPU(1)
oModel.setTemperature(0.6)
oModel.setMaxTokens(256)

cPrompt = "Solve step by step: A train travels at 120 km/h and covers a distance of 300 km. How many minutes did the trip take?"
see nl + "========================================================" + nl
see "Prompt: " + cPrompt + nl
see "========================================================" + nl

tGen0 = clock()
cAnswer = oModel.generate(cPrompt)
tGen1 = clock()

see nl + cAnswer + nl
see "========================================================" + nl
see f"Generation Speed: {oModel.tokensPerSecond()} tokens/sec (Latency: {(tGen1 - tGen0)/1000.0} sec)" + nl

## ✒️ Step 6: Hybrid RAG & Knowledge Retrieval via AlQalam C++20
The **AlQalam** extension blends dense vector semantic search (70%) with sub-millisecond lexical inverted indexing `QalamIndex` (30%) and zero-overhead binary persistence (`saveBinary` / `loadBinary`).

In [ ]:
%%madar
load "stdlibCore.ring"
load "libraries/madar_ai/madar_ai.ring"
load "libraries/madar_ai/rag.ring"
load "alqalam.ring"

cModelPath = "models/deepseek-r1-distill-qwen-7b-q4_0.gguf"
if not fexists(cModelPath) cModelPath = "models/qwen2.5-7b-instruct-q4_0.gguf" ok

oTimer = new QalamChronos()
oTimer.reset()

see "[RAG] Initializing language model for embedding and generation... " + nl
llm = ai_open(cModelPath)
llm.setGPU(1)
llm.setThreads(4)
store = new VectorStore(llm)

# Index reference knowledge chunks
store.addText("Madar runtime supports GPU hardware acceleration via zero-copy OpenCL alongside AVX2 SIMD dot-product kernels.")
store.addText("The AlQalam library provides high-speed C++ data structures including QalamIndex for O(1) lexical indexing.")
store.addText("Damascus is the capital of the Syrian Arab Republic and is recognized as one of the oldest continuously inhabited cities.")

cQuestion = "What are the core acceleration features of Madar runtime?"
see nl + "Query: " + cQuestion + nl

# Execute hybrid retrieval (70% Semantic + 30% Lexical via AlQalam)
oTimer.reset()
aResults = store.hybridSearch(cQuestion, 2, 0.7)
see f"[RAG] Hybrid search finished in {oTimer.elapsed()}:" + nl

for res in aResults
    see f"  - [Score: {res[1]}] (Semantic: {res[3]} | Lexical: {res[4]}) {res[2].cText}" + nl
next

best_text = aResults[1][2].cText
see nl + "[RAG] Synthesizing factual answer grounded in retrieved chunk:" + nl
cPrompt = f"Answer concisely based strictly on this reference: {best_text}\nQuestion: {cQuestion}"
see llm.generate(cPrompt) + nl

## 🤖 Step 7: Autonomous AI Agent with Tool Execution & Vector Memory
The Madar Autonomous Agent (`AIAgent`) automatically intercepts `<tool_call>`, executes native Ring/C functions (mathematics, file access, system diagnostics, and DataFrame analytics), injects `<tool_response>`, and preserves long-term episodic memory.

In [ ]:
%%madar
load "stdlibCore.ring"
load "libraries/madar_ai/agent.ring"

cModelPath = "models/deepseek-r1-distill-qwen-7b-q4_0.gguf"
if not fexists(cModelPath)
    cModelPath = "models/qwen2.5-7b-instruct-q4_0.gguf"
ok

oAgent = new AIAgent(cModelPath)
oAgent.oModel.setGPU(1)
oAgent.oModel.setThreads(4)

# Inscribe permanent facts into the agent's vector memory
oAgent.remember("Madar is an AI-native, high-performance dialect and runtime for the Ring ecosystem.")
oAgent.remember("Madar is implemented in pure ANSI C/C99 with AVX2 and OpenCL hardware acceleration.")

see "[AGENT] Vector memory initialized! Submitting multi-step query:" + nl

cQuery = "Calculate 4589 * 321 using the calc tool, and summarize what you remember about Madar runtime."
see nl + "User > " + cQuery + nl + nl

cResponse = oAgent.chat(cQuery)
see nl + "Agent > " + nl + cResponse + nl

## 💬 Step 8: Interactive GUI Chat Widget
You can interactively chat with the model directly inside Colab using this clean `ipywidgets` user interface.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import subprocess, os, tempfile

chat_history = []

out_area = widgets.Output()
text_input = widgets.Text(
    placeholder='Type your question for Madar here and press Enter...',
    layout=widgets.Layout(width='75%')
)
send_btn = widgets.Button(
    description='Send 🚀',
    button_style='primary',
    layout=widgets.Layout(width='20%')
)

def ask_madar(prompt):
    ring_code = f'''
load "libraries/madar_ai/madar_ai.ring"
cModel = "models/deepseek-r1-distill-qwen-7b-q4_0.gguf"
if not fexists(cModel) cModel = "models/qwen2.5-7b-instruct-q4_0.gguf" ok
model = ai_open(cModel)
model.setGPU(1)
model.setThreads(4)
model.setTemperature(0.7)
model.setMaxTokens(200)
see model.generate("{prompt}")
'''
    with tempfile.NamedTemporaryFile("w", suffix=".ring", delete=False, encoding="utf-8") as f:
        f.write(ring_code)
        tmp_name = f.name
    
    try:
        env = os.environ.copy()
        env["LD_LIBRARY_PATH"] = os.path.abspath("lib") + ":" + env.get("LD_LIBRARY_PATH", "")
        res = subprocess.run([os.path.abspath("bin/ring"), tmp_name], capture_output=True, text=True, env=env)
        return res.stdout.strip()
    finally:
        if os.path.exists(tmp_name):
            os.remove(tmp_name)

def on_send(b):
    user_msg = text_input.value.strip()
    if not user_msg:
        return
    text_input.value = ''
    
    with out_area:
        display(HTML(f'<div style="background-color:#e3f2fd; padding:10px; border-radius:8px; margin:5px 0;"><b>🧑 You:</b> {user_msg}</div>'))
        display(HTML('<div id="loading" style="color:#888;"><i>⏳ Madar is thinking...</i></div>'))
    
    bot_msg = ask_madar(user_msg)
    
    with out_area:
        clear_output(wait=True)
        for role, text in chat_history:
            bg = '#e3f2fd' if role == 'user' else '#f1f8e9'
            author = '🧑 You' if role == 'user' else '🪐 Madar'
            display(HTML(f'<div style="background-color:{bg}; padding:10px; border-radius:8px; margin:5px 0;"><b>{author}:</b> {text}</div>'))
        
        display(HTML(f'<div style="background-color:#f1f8e9; padding:10px; border-radius:8px; margin:5px 0;"><b>🪐 Madar:</b> {bot_msg}</div>'))
    
    chat_history.append(('user', user_msg))
    chat_history.append(('bot', bot_msg))

send_btn.on_click(on_send)
text_input.on_submit(on_send)

controls = widgets.HBox([text_input, send_btn])
display(widgets.VBox([out_area, controls]))